# Day 7: TRIBE v2 Feature Extraction → fMRI Encoding with Memory
**Goal:** Connect TRIBE v2's pretrained feature extractors (video, audio, text) to generate stimulus embeddings, then train the memory-augmented encoder to predict fMRI from these features.

**Key Insight from Day 6:** Training with fMRI→fMRI gives R=0.944 but memory can't help because the model reconstructs current windows from itself. The REAL encoding task is: stimulus features → fMRI, where temporal context (memory) matters.

**Architecture:**
```
Video frames → CLIP ViT-L/14 ──────────┐
Audio waveform → Whisper large-v2 ─────┼→ Fused Feature → Memory Module → Predicted fMRI
Text/subtitles → LLaMA 3.2 1B ────────┘
```

**Runtime:** A100 GPU

In [ ]:
# === INSTALL DEPENDENCIES ===
!pip install torch torchvision torchaudio -q
!pip install transformers>=4.36.0 accelerate safetensors -q
!pip install nibabel nilearn scipy matplotlib seaborn -q
!pip install tqdm einops h5py scikit-learn -q

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import json
import time
from tqdm.auto import tqdm

PROJECT_DIR = '/content/drive/MyDrive/Research/memory-brain-encoding'
RESULTS_DIR = os.path.join(PROJECT_DIR, 'day7_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Results dir: {RESULTS_DIR}")

---
## 1. Load TRIBE v2 Feature Extractors

TRIBE v2 uses three pretrained encoders to extract stimulus representations:
- **Video:** CLIP ViT-L/14 → 768-dim per frame
- **Audio:** Whisper large-v2 (encoder only) → 1280-dim per segment
- **Text:** LLaMA 3.2 1B (or BERT-large fallback) → 2048-dim per token

We extract features at TR resolution (1.49s) and concatenate them into a single multimodal embedding.

In [ ]:
from transformers import (
    CLIPModel, CLIPProcessor,
    WhisperModel, WhisperFeatureExtractor,
    AutoModel, AutoTokenizer
)

class TRIBEv2FeatureExtractor(nn.Module):
    """
    Loads TRIBE v2's three pretrained feature encoders.
    Extracts video/audio/text features and fuses them via concatenation.
    """

    def __init__(self, device='cuda', use_llama=True):
        super().__init__()
        self.device = device
        self.feature_dims = {}

        print("Loading TRIBE v2 feature extractors...")

        # Video: CLIP ViT-L/14
        print("  Loading CLIP ViT-L/14 for video...")
        self.clip_model = CLIPModel.from_pretrained(
            "openai/clip-vit-large-patch14"
        ).to(device)
        self.clip_processor = CLIPProcessor.from_pretrained(
            "openai/clip-vit-large-patch14"
        )
        self.clip_model.eval()
        self.feature_dims['video'] = 768
        print(f"    Done: {self.feature_dims['video']}-dim")

        # Audio: Whisper large-v2 encoder
        print("  Loading Whisper large-v2 for audio...")
        self.whisper_model = WhisperModel.from_pretrained(
            "openai/whisper-large-v2"
        ).encoder.to(device)
        self.whisper_fe = WhisperFeatureExtractor.from_pretrained(
            "openai/whisper-large-v2"
        )
        self.whisper_model.eval()
        self.feature_dims['audio'] = 1280
        print(f"    Done: {self.feature_dims['audio']}-dim")

        # Text: LLaMA 3.2 1B or BERT fallback
        if use_llama:
            try:
                print("  Loading LLaMA 3.2 1B for text...")
                self.text_model = AutoModel.from_pretrained(
                    "meta-llama/Llama-3.2-1B",
                    torch_dtype=torch.float16,
                    device_map="auto"
                )
                self.text_tokenizer = AutoTokenizer.from_pretrained(
                    "meta-llama/Llama-3.2-1B"
                )
                if self.text_tokenizer.pad_token is None:
                    self.text_tokenizer.pad_token = self.text_tokenizer.eos_token
                self.feature_dims['text'] = 2048
                self.text_model_name = 'llama-3.2-1b'
                print(f"    Done: {self.feature_dims['text']}-dim")
            except Exception as e:
                print(f"    LLaMA failed ({e}), falling back to BERT...")
                use_llama = False

        if not use_llama:
            print("  Loading BERT-large for text (fallback)...")
            self.text_model = AutoModel.from_pretrained(
                "bert-large-uncased"
            ).to(device)
            self.text_tokenizer = AutoTokenizer.from_pretrained(
                "bert-large-uncased"
            )
            self.feature_dims['text'] = 1024
            self.text_model_name = 'bert-large'
            print(f"    Done: {self.feature_dims['text']}-dim")

        self.text_model.eval()
        self.total_dim = sum(self.feature_dims.values())
        print(f"\nTotal fused feature dim: {self.total_dim}")
        print(f"  video={self.feature_dims['video']} + "
              f"audio={self.feature_dims['audio']} + "
              f"text={self.feature_dims['text']}")

    @torch.no_grad()
    def extract_video_features(self, frames):
        """Extract CLIP features from video frames. Returns [n_TRs, 768]."""
        features = []
        for i in range(0, len(frames), 16):
            batch = frames[i:i+16]
            inputs = self.clip_processor(images=batch, return_tensors="pt").to(self.device)
            outputs = self.clip_model.get_image_features(**inputs)
            outputs = outputs / outputs.norm(dim=-1, keepdim=True)
            features.append(outputs.cpu())
        return torch.cat(features, dim=0)

    @torch.no_grad()
    def extract_audio_features(self, waveform, sr=16000, tr_duration=1.49):
        """Extract Whisper encoder features from audio. Returns [n_TRs, 1280]."""
        samples_per_tr = int(sr * tr_duration)
        n_trs = len(waveform) // samples_per_tr
        features = []
        for i in range(n_trs):
            segment = waveform[i*samples_per_tr:(i+1)*samples_per_tr]
            padded = np.zeros(sr * 30)
            padded[:len(segment)] = segment
            inputs = self.whisper_fe(padded, sampling_rate=sr, return_tensors="pt").input_features.to(self.device)
            outputs = self.whisper_model(inputs)
            feat = outputs.last_hidden_state.mean(dim=1)
            features.append(feat.cpu())
        return torch.cat(features, dim=0)

    @torch.no_grad()
    def extract_text_features(self, texts):
        """Extract text features from subtitle segments. Returns [n_TRs, text_dim]."""
        features = []
        for i in range(0, len(texts), 32):
            batch = [t if t.strip() else "[silence]" for t in texts[i:i+32]]
            inputs = self.text_tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=128
            ).to(self.device)
            outputs = self.text_model(**inputs)
            hidden = outputs.last_hidden_state
            mask = inputs['attention_mask'].unsqueeze(-1)
            feat = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            features.append(feat.float().cpu())
        return torch.cat(features, dim=0)

    def fuse_features(self, video_feats, audio_feats, text_feats):
        """Concatenation fusion. Returns [n_TRs, total_dim]."""
        min_len = min(len(video_feats), len(audio_feats), len(text_feats))
        return torch.cat([
            video_feats[:min_len],
            audio_feats[:min_len],
            text_feats[:min_len]
        ], dim=-1)

# Initialize
feature_extractor = TRIBEv2FeatureExtractor(device=device, use_llama=True)
print(f"\nAll feature extractors loaded!")

---
## 2. Synthetic Stimulus Data with Temporal Dependencies

Before processing real video, we validate the full pipeline with synthetic features that have **known** temporal structure. This includes:
- **Narrative events** with consistent feature templates
- **Exponential memory decay** in encoding weights
- **Narrative callbacks** in hippocampal vertices (later events referencing earlier ones)

If memory doesn't help here, something is broken.

In [ ]:
from sklearn.decomposition import PCA

def generate_synthetic_stimulus_features(
    n_trs=300, feature_dim=None, n_vertices=20484,
    memory_dependency_strength=0.5, n_narrative_events=5, seed=42
):
    """
    Generate synthetic stimulus->fMRI data where fMRI depends on BOTH
    current stimulus AND past context (simulating narrative memory).
    Includes 'narrative callback' effects in hippocampal vertices.
    """
    if feature_dim is None:
        feature_dim = feature_extractor.total_dim

    np.random.seed(seed)
    torch.manual_seed(seed)

    # Create stimulus features with narrative event structure
    event_boundaries = np.sort(np.random.choice(
        range(20, n_trs-20), n_narrative_events-1, replace=False
    ))
    event_boundaries = np.concatenate([[0], event_boundaries, [n_trs]])

    features = np.zeros((n_trs, feature_dim))
    event_templates = np.random.randn(n_narrative_events, feature_dim) * 0.5

    for i in range(n_narrative_events):
        start, end = event_boundaries[i], event_boundaries[i+1]
        segment_len = end - start
        t = np.linspace(0, 1, segment_len)[:, None]
        features[start:end] = (
            event_templates[i]
            + np.random.randn(segment_len, feature_dim) * 0.3
            + np.sin(2 * np.pi * t * np.random.uniform(0.5, 3)) * 0.2
        )

    # Encoding weights per vertex
    W_current = np.random.randn(feature_dim, n_vertices) * 0.1
    W_memory = np.random.randn(feature_dim, n_vertices) * 0.1

    hrf_delay = 3
    fmri = np.zeros((n_trs, n_vertices))
    memory_state = np.zeros(feature_dim)
    decay = 0.85

    for t in range(n_trs):
        current_contribution = features[t] @ W_current
        memory_state = decay * memory_state + (1-decay) * features[t]
        memory_contribution = memory_state @ W_memory
        fmri_t = (
            (1 - memory_dependency_strength) * current_contribution
            + memory_dependency_strength * memory_contribution
        )
        target_t = min(t + hrf_delay, n_trs - 1)
        fmri[target_t] += fmri_t

    # Narrative callbacks in hippocampal region
    hippocampal_vertices = list(range(0, 500))
    for callback_t in np.random.choice(range(n_trs//2, n_trs), 10, replace=False):
        ref_event = np.random.randint(0, n_narrative_events//2)
        ref_feature = features[event_boundaries[ref_event]]
        reinstatement = ref_feature @ W_memory[:, hippocampal_vertices]
        fmri[callback_t, hippocampal_vertices] += reinstatement * 0.3

    # Normalize
    fmri += np.random.randn(*fmri.shape) * 0.5
    fmri = (fmri - fmri.mean(axis=0)) / (fmri.std(axis=0) + 1e-8)
    features = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-8)

    return {
        'features': torch.FloatTensor(features),
        'fmri': torch.FloatTensor(fmri),
        'event_boundaries': event_boundaries,
        'hippocampal_vertices': hippocampal_vertices,
        'n_vertices': n_vertices,
        'feature_dim': feature_dim,
        'memory_dependency_strength': memory_dependency_strength,
    }

synth_data = generate_synthetic_stimulus_features(
    n_trs=300, memory_dependency_strength=0.5, n_narrative_events=5
)

print(f"Stimulus features: {synth_data['features'].shape}")
print(f"fMRI targets: {synth_data['fmri'].shape}")
print(f"Narrative events: {len(synth_data['event_boundaries'])-1}")
print(f"Hippocampal ROI: {len(synth_data['hippocampal_vertices'])} vertices")

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

pca = PCA(n_components=3)
feat_pcs = pca.fit_transform(synth_data['features'].numpy())
for i in range(3):
    axes[0].plot(feat_pcs[:, i], alpha=0.7, label=f'PC{i+1}')
for b in synth_data['event_boundaries'][1:-1]:
    axes[0].axvline(b, color='red', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Stimulus Features')
axes[0].legend(loc='upper right')
axes[0].set_title('Synthetic Stimulus -> fMRI with Temporal Dependencies')

hippo_fmri = synth_data['fmri'][:, synth_data['hippocampal_vertices'][:20]].numpy()
axes[1].imshow(hippo_fmri.T, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
axes[1].set_ylabel('Hippocampal\nVertices')

cortical_fmri = synth_data['fmri'][:, 500:520].numpy()
axes[2].imshow(cortical_fmri.T, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
axes[2].set_ylabel('Cortical\nVertices')
axes[2].set_xlabel('TR')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'day7_synthetic_data_structure.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: day7_synthetic_data_structure.png")

---
## 3. Encoding Model: Stimulus Features → fMRI (with Memory)

The core architecture has three stages:
1. **FeatureProjector** — maps concatenated multimodal features (4096-dim) to 512-dim embedding
2. **MemoryModule** — circular buffer + cross-attention with learnable gate (starts near 0)
3. **Cortical Decoder** — predicts fMRI for all 20,484 vertices

We create two versions (with and without memory) for controlled comparison.

In [ ]:
class FeatureProjector(nn.Module):
    """Projects concatenated features to common embedding space."""
    def __init__(self, input_dim, hidden_dim=512, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class MemoryModule(nn.Module):
    """
    Circular buffer + cross-attention memory.
    Current embedding queries past embeddings via multi-head attention.
    Learnable gate controls memory influence (initialized near 0).
    """
    def __init__(self, embed_dim=512, memory_size=64, n_heads=8, dropout=0.1):
        super().__init__()
        self.memory_size = memory_size
        self.embed_dim = embed_dim
        self.cross_attn = nn.MultiheadAttention(embed_dim, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim * 4, embed_dim), nn.Dropout(dropout),
        )
        self.gate = nn.Parameter(torch.tensor(-2.0))
        # Memory state as plain Python (not registered buffers)
        self.memory_buffer = None
        self.memory_ptr = 0
        self.memory_count = 0

    def reset_memory(self, batch_size=1, device=None):
        if device is None:
            device = self.gate.device
        self.memory_buffer = torch.zeros(batch_size, self.memory_size, self.embed_dim, device=device)
        self.memory_ptr = 0
        self.memory_count = 0

    def write_memory(self, x):
        # Clone to avoid in-place modification error with autograd
        new_buffer = self.memory_buffer.clone()
        new_buffer[:, self.memory_ptr] = x.detach()
        self.memory_buffer = new_buffer
        self.memory_ptr = (self.memory_ptr + 1) % self.memory_size
        self.memory_count = min(self.memory_count + 1, self.memory_size)

    def forward(self, x):
        gate = torch.sigmoid(self.gate)

        if self.memory_buffer is None or self.memory_count == 0:
            self.write_memory(x)
            return x

        n_valid = self.memory_count
        memory = self.memory_buffer[:, :n_valid]

        query = x.unsqueeze(1)
        attn_out, _ = self.cross_attn(query, memory, memory)
        attn_out = attn_out.squeeze(1)

        x_mem = self.norm1(x + gate * attn_out)
        x_mem = self.norm2(x_mem + self.ffn(x_mem))

        self.write_memory(x)
        return x_mem


class StimulusToFMRIEncoder(nn.Module):
    """
    Full encoding pipeline: multimodal features -> predicted fMRI.
    Can run with or without memory for controlled comparison.
    """
    def __init__(self, feature_dim, n_vertices=20484, hidden_dim=512,
                 memory_size=64, n_heads=8, use_memory=True, dropout=0.1):
        super().__init__()
        self.use_memory = use_memory
        self.projector = FeatureProjector(feature_dim, hidden_dim, dropout)
        self.memory = MemoryModule(hidden_dim, memory_size, n_heads, dropout) if use_memory else None
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2), nn.LayerNorm(hidden_dim * 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, n_vertices),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def reset_memory(self, batch_size=1):
        if self.memory is not None:
            self.memory.reset_memory(batch_size, device=next(self.parameters()).device)

    def forward(self, features):
        embed = self.projector(features)
        if self.memory is not None:
            embed = self.memory(embed)
        return self.decoder(embed)

# Build both models
feature_dim = feature_extractor.total_dim
n_vertices = synth_data['n_vertices']

model_with_memory = StimulusToFMRIEncoder(
    feature_dim=feature_dim, n_vertices=n_vertices,
    hidden_dim=512, memory_size=64, use_memory=True
).to(device)

model_without_memory = StimulusToFMRIEncoder(
    feature_dim=feature_dim, n_vertices=n_vertices,
    hidden_dim=512, use_memory=False
).to(device)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"Model WITH memory:    {count_params(model_with_memory):,} params")
print(f"Model WITHOUT memory: {count_params(model_without_memory):,} params")
print(f"Memory module adds:   {count_params(model_with_memory) - count_params(model_without_memory):,} params")
print(f"Memory gate init:     {torch.sigmoid(model_with_memory.memory.gate).item():.4f}")

---
## 4. Training: Stimulus → fMRI Encoding (Memory vs No Memory)

We train both models on the synthetic data, processing TRs **sequentially** so the memory module can maintain state. We track correlations separately for hippocampal vs cortical ROIs.

In [ ]:
from torch.utils.data import Dataset

class TemporalEncodingDataset(Dataset):
    """Yields contiguous temporal windows for sequential processing."""
    def __init__(self, features, fmri, seq_len=50, stride=25):
        self.features = features
        self.fmri = fmri
        self.seq_len = seq_len
        self.stride = stride
        self.n_sequences = max(1, (len(features) - seq_len) // stride + 1)
    def __len__(self):
        return self.n_sequences
    def __getitem__(self, idx):
        start = idx * self.stride
        end = start + self.seq_len
        return self.features[start:end], self.fmri[start:end]


def vertex_correlations(preds, targets, vertex_indices=None):
    """Compute mean Pearson R across vertices."""
    if vertex_indices is not None:
        preds = preds[:, vertex_indices]
        targets = targets[:, vertex_indices]
    corrs = []
    for v in range(preds.shape[1]):
        if targets[:, v].std() > 1e-6:
            r = np.corrcoef(preds[:, v], targets[:, v])[0, 1]
            if not np.isnan(r):
                corrs.append(r)
    return np.mean(corrs) if corrs else 0.0


def train_encoding_model(model, train_feat, train_fmri, val_feat, val_fmri,
                          n_epochs=50, lr=1e-4, seq_len=50, model_name="model",
                          hippocampal_vertices=None):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)
    train_ds = TemporalEncodingDataset(train_feat, train_fmri, seq_len)
    val_ds = TemporalEncodingDataset(val_feat, val_fmri, seq_len)

    history = {
        'train_loss': [], 'val_loss': [],
        'val_corr_all': [], 'val_corr_hippo': [], 'val_corr_cortex': [],
        'gate_values': []
    }

    for epoch in range(n_epochs):
        # Train
        model.train()
        epoch_loss, n_batches = 0, 0
        for seq_f, seq_t in train_ds:
            seq_f, seq_t = seq_f.to(device), seq_t.to(device)
            model.reset_memory(batch_size=1)
            optimizer.zero_grad()
            seq_loss = 0
            for t in range(len(seq_f)):
                pred = model(seq_f[t:t+1])
                seq_loss += F.mse_loss(pred, seq_t[t:t+1])
            seq_loss /= len(seq_f)
            seq_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += seq_loss.item()
            n_batches += 1

        # Validate
        model.eval()
        val_preds, val_tgts = [], []
        with torch.no_grad():
            for seq_f, seq_t in val_ds:
                seq_f = seq_f.to(device)
                model.reset_memory(batch_size=1)
                for t in range(len(seq_f)):
                    val_preds.append(model(seq_f[t:t+1]).cpu())
                    val_tgts.append(seq_t[t:t+1])

        val_preds = torch.cat(val_preds, 0).numpy()
        val_tgts = torch.cat(val_tgts, 0).numpy()
        val_loss = np.mean((val_preds - val_tgts) ** 2)

        corr_all = vertex_correlations(val_preds, val_tgts)
        corr_hippo = vertex_correlations(val_preds, val_tgts, hippocampal_vertices) if hippocampal_vertices else 0.0
        corr_cortex = vertex_correlations(val_preds, val_tgts, list(range(500, min(1000, val_preds.shape[1]))))

        gate_val = torch.sigmoid(model.memory.gate).item() if model.memory else 0.0

        history['train_loss'].append(epoch_loss / max(n_batches, 1))
        history['val_loss'].append(val_loss)
        history['val_corr_all'].append(corr_all)
        history['val_corr_hippo'].append(corr_hippo)
        history['val_corr_cortex'].append(corr_cortex)
        history['gate_values'].append(gate_val)

        scheduler.step()

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"[{model_name}] Epoch {epoch+1:3d} | "
                  f"Train: {history['train_loss'][-1]:.4f} | Val: {val_loss:.4f} | "
                  f"R(all): {corr_all:.4f} | R(hippo): {corr_hippo:.4f} | "
                  f"R(cortex): {corr_cortex:.4f}"
                  + (f" | Gate: {gate_val:.4f}" if gate_val > 0 else ""))

    return history

In [ ]:
# Split: 70% train, 30% val
n_train = int(0.7 * len(synth_data['features']))
train_feat = synth_data['features'][:n_train]
train_fmri = synth_data['fmri'][:n_train]
val_feat = synth_data['features'][n_train:]
val_fmri = synth_data['fmri'][n_train:]

print("=" * 70)
print("Training WITH memory...")
print("=" * 70)
history_mem = train_encoding_model(
    model_with_memory, train_feat, train_fmri, val_feat, val_fmri,
    n_epochs=50, lr=1e-4, model_name="Memory",
    hippocampal_vertices=synth_data['hippocampal_vertices']
)

print()
print("=" * 70)
print("Training WITHOUT memory...")
print("=" * 70)
history_nomem = train_encoding_model(
    model_without_memory, train_feat, train_fmri, val_feat, val_fmri,
    n_epochs=50, lr=1e-4, model_name="NoMem",
    hippocampal_vertices=synth_data['hippocampal_vertices']
)

---
## 5. Results: Memory vs No Memory Comparison

Does the memory module improve encoding accuracy? We expect the biggest benefit in **hippocampal** regions where we embedded narrative callback effects.

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)

# Training curves
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history_mem['train_loss'], label='Memory (train)', color='#2196F3')
ax1.plot(history_mem['val_loss'], label='Memory (val)', color='#2196F3', linestyle='--')
ax1.plot(history_nomem['train_loss'], label='No Memory (train)', color='#FF5722')
ax1.plot(history_nomem['val_loss'], label='No Memory (val)', color='#FF5722', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE Loss')
ax1.set_title('Training Curves'); ax1.legend(fontsize=8); ax1.set_yscale('log')

# Overall correlation
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(history_mem['val_corr_all'], label='Memory', color='#2196F3', linewidth=2)
ax2.plot(history_nomem['val_corr_all'], label='No Memory', color='#FF5722', linewidth=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Mean R')
ax2.set_title('Overall Encoding Accuracy'); ax2.legend()

# Hippocampal vs Cortical
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(history_mem['val_corr_hippo'], label='Memory (hippo)', color='#2196F3', linewidth=2)
ax3.plot(history_nomem['val_corr_hippo'], label='No Memory (hippo)', color='#FF5722', linewidth=2)
ax3.plot(history_mem['val_corr_cortex'], label='Memory (cortex)', color='#2196F3', linewidth=2, linestyle='--')
ax3.plot(history_nomem['val_corr_cortex'], label='No Memory (cortex)', color='#FF5722', linewidth=2, linestyle='--')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('Mean R')
ax3.set_title('Hippocampal vs Cortical'); ax3.legend(fontsize=7)

# Gate evolution
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(history_mem['gate_values'], color='#4CAF50', linewidth=2)
ax4.set_xlabel('Epoch'); ax4.set_ylabel('Gate (sigma)')
ax4.set_title('Memory Gate Opening')
ax4.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax4.set_ylim(-0.05, 1.05)

# Bar chart
ax5 = fig.add_subplot(gs[1, 1])
metrics = {
    'All': [history_mem['val_corr_all'][-1], history_nomem['val_corr_all'][-1]],
    'Hippo': [history_mem['val_corr_hippo'][-1], history_nomem['val_corr_hippo'][-1]],
    'Cortex': [history_mem['val_corr_cortex'][-1], history_nomem['val_corr_cortex'][-1]],
}
x = np.arange(len(metrics)); w = 0.35
b1 = ax5.bar(x - w/2, [v[0] for v in metrics.values()], w, label='Memory', color='#2196F3', alpha=0.8)
b2 = ax5.bar(x + w/2, [v[1] for v in metrics.values()], w, label='No Memory', color='#FF5722', alpha=0.8)
ax5.set_xticks(x); ax5.set_xticklabels(metrics.keys())
ax5.set_ylabel('Correlation (R)'); ax5.set_title('Final Performance'); ax5.legend()
for bar in b1:
    ax5.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{bar.get_height():.3f}', ha='center', fontsize=8)
for bar in b2:
    ax5.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{bar.get_height():.3f}', ha='center', fontsize=8)

# Summary
ax6 = fig.add_subplot(gs[1, 2]); ax6.axis('off')
d_all = history_mem['val_corr_all'][-1] - history_nomem['val_corr_all'][-1]
d_hip = history_mem['val_corr_hippo'][-1] - history_nomem['val_corr_hippo'][-1]
d_ctx = history_mem['val_corr_cortex'][-1] - history_nomem['val_corr_cortex'][-1]
summary = (f"Day 7 Summary\n{'='*25}\n\n"
           f"Memory Benefit (delta R):\n  All:   {d_all:+.4f}\n  Hippo: {d_hip:+.4f}\n  Cortex: {d_ctx:+.4f}\n\n"
           f"Gate: {history_mem['gate_values'][-1]:.4f}\n\n"
           f"Params:\n  w/ mem:  {count_params(model_with_memory):,}\n  w/o mem: {count_params(model_without_memory):,}")
ax6.text(0.05, 0.95, summary, transform=ax6.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.3))

plt.suptitle('Day 7: Stimulus to fMRI Encoding with Memory Augmentation', fontsize=14, fontweight='bold', y=1.02)
plt.savefig(os.path.join(RESULTS_DIR, 'day7_memory_vs_nomemory_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nHippocampal memory benefit: delta R = {d_hip:+.4f}")

---
## 6. Check for Real Algonauts Data

If you've uploaded the Friends fMRI and stimulus files to Google Drive, this cell will find them. Otherwise it shows what's needed for Day 8.

In [ ]:
import glob

algonauts_paths = [
    '/content/drive/MyDrive/Research/memory-brain-encoding/algonauts_data',
    '/content/drive/MyDrive/Research/memory-brain-encoding/data',
    '/content/drive/MyDrive/algonauts',
]

data_path = None
for p in algonauts_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path:
    print(f"Found data at: {data_path}")
    print(f"Contents: {os.listdir(data_path)[:20]}")
    fmri_files = (glob.glob(os.path.join(data_path, '**/*.npy'), recursive=True)
                  + glob.glob(os.path.join(data_path, '**/*.nii*'), recursive=True)
                  + glob.glob(os.path.join(data_path, '**/*.h5'), recursive=True))
    print(f"fMRI files: {len(fmri_files)}")
    stim_files = (glob.glob(os.path.join(data_path, '**/*.mp4'), recursive=True)
                  + glob.glob(os.path.join(data_path, '**/*.mkv'), recursive=True)
                  + glob.glob(os.path.join(data_path, '**/*.wav'), recursive=True))
    print(f"Stimulus files: {len(stim_files)}")
else:
    print("Algonauts data not found on Drive.")
    print("\nTo use real data for Day 8, upload to:")
    print("  Drive > Research > memory-brain-encoding > algonauts_data/")
    print("\nYou already have DataLad-downloaded data on your Mac.")
    print("Upload the friends_s01 fMRI and stimulus segments to Drive.")

---
## 7. Save Models and Results

In [ ]:
# Save model checkpoints
torch.save({
    'model_state_dict': model_with_memory.state_dict(),
    'config': {'feature_dim': feature_dim, 'n_vertices': n_vertices,
               'hidden_dim': 512, 'memory_size': 64, 'use_memory': True},
    'history': history_mem,
    'final_gate': torch.sigmoid(model_with_memory.memory.gate).item(),
}, os.path.join(RESULTS_DIR, 'day7_model_with_memory.pt'))

torch.save({
    'model_state_dict': model_without_memory.state_dict(),
    'config': {'feature_dim': feature_dim, 'n_vertices': n_vertices,
               'hidden_dim': 512, 'use_memory': False},
    'history': history_nomem,
}, os.path.join(RESULTS_DIR, 'day7_model_without_memory.pt'))

# Save results JSON
d_all = history_mem['val_corr_all'][-1] - history_nomem['val_corr_all'][-1]
d_hip = history_mem['val_corr_hippo'][-1] - history_nomem['val_corr_hippo'][-1]
d_ctx = history_mem['val_corr_cortex'][-1] - history_nomem['val_corr_cortex'][-1]

results = {
    'day': 7,
    'date': time.strftime('%Y-%m-%d'),
    'task': 'Stimulus -> fMRI encoding with memory augmentation',
    'data_type': 'synthetic (pipeline validation)',
    'feature_extractors': {
        'video': 'CLIP ViT-L/14 (768-dim)',
        'audio': 'Whisper large-v2 (1280-dim)',
        'text': f'{feature_extractor.text_model_name} ({feature_extractor.feature_dims["text"]}-dim)',
        'total_dim': feature_extractor.total_dim,
    },
    'with_memory': {
        'params': count_params(model_with_memory),
        'final_gate': history_mem['gate_values'][-1],
        'R_all': history_mem['val_corr_all'][-1],
        'R_hippo': history_mem['val_corr_hippo'][-1],
        'R_cortex': history_mem['val_corr_cortex'][-1],
    },
    'without_memory': {
        'params': count_params(model_without_memory),
        'R_all': history_nomem['val_corr_all'][-1],
        'R_hippo': history_nomem['val_corr_hippo'][-1],
        'R_cortex': history_nomem['val_corr_cortex'][-1],
    },
    'memory_benefit': {'delta_all': d_all, 'delta_hippo': d_hip, 'delta_cortex': d_ctx},
    'next_steps': [
        'Day 8: Extract real TRIBE v2 features from Friends video/audio/text',
        'Day 9: Train on real Algonauts/CNeuroMod fMRI data',
        'Day 10: ROI analysis — hippocampus, PFC, TPJ',
    ],
}

with open(os.path.join(RESULTS_DIR, 'day7_results.json'), 'w') as f:
    json.dump(results, f, indent=2, default=str)

print("Saved to Google Drive:")
print(f"  {RESULTS_DIR}/")
print(f"  ├── day7_model_with_memory.pt")
print(f"  ├── day7_model_without_memory.pt")
print(f"  ├── day7_results.json")
print(f"  ├── day7_synthetic_data_structure.png")
print(f"  └── day7_memory_vs_nomemory_comparison.png")
print()
print("=" * 60)
print("DAY 7 COMPLETE")
print("=" * 60)
print(f"\n1. Loaded TRIBE v2 extractors (CLIP + Whisper + {feature_extractor.text_model_name})")
print(f"2. Built stimulus->fMRI encoder with gated memory")
print(f"3. Validated on synthetic temporal data")
print(f"4. Hippocampal memory benefit: delta R = {d_hip:+.4f}")
print(f"5. Gate opened: 0 -> {history_mem['gate_values'][-1]:.3f}")
print(f"\nReady for Day 8: Real feature extraction from Friends episodes!")